# 07 - Sample base 8x on GSM8K train

## 1. Installations

In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.11.2', 'triton') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
try:
    import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
except Exception:
    _torch_v = ()
_torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
!uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq math-verify
print("is_t4:", is_t4, "| vllm:", _vllm)

## 2. Environment check

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["USE_TF"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import importlib.metadata as md
for pkg in ["torch", "vllm", "transformers", "trl", "peft", "unsloth", "datasets", "math-verify"]:
    try: print(f"{pkg:13s} {md.version(pkg)}")
    except md.PackageNotFoundError: print(f"{pkg:13s} NOT INSTALLED")

import torch
print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3. Shared extractor + eval helpers

In [ ]:
%%writefile rf_extract.py
import re

BOXED_SUFFIX = "\nPlease reason step by step, and put your final answer within \\boxed{}."


def user_prompt(question, style="boxed"):
    if style == "boxed":
        return question + BOXED_SUFFIX
    if style == "plain":
        return question
    raise ValueError(style)


def _boxed_spans(text):
    key = r"\boxed{"
    spans, i = [], 0
    while True:
        j = text.find(key, i)
        if j == -1:
            break
        k = j + len(key)
        start, depth = k, 1
        while k < len(text) and depth > 0:
            if text[k] == "{":
                depth += 1
            elif text[k] == "}":
                depth -= 1
            k += 1
        spans.append(text[start:k - 1])
        i = k
    return spans


def _unwrap_text(s):
    return re.sub(r"\\(?:text|textbf|mbox|mathrm)\{([^{}]*)\}", r"\1", s)


def norm(x):
    if x is None:
        return None
    s = str(x).strip()
    s = _unwrap_text(s)
    s = s.replace("\\!", "").replace("\\,", "").replace("\\;", "").replace(" ", "")
    s = s.replace("\\$", "").replace("$", "")
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)
    s = s.replace("\\%", "").replace("%", "")
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("^\\circ", "").replace("^{\\circ}", "")
    s = s.rstrip(".").strip()
    if s.startswith("{") and s.endswith("}"):
        s = s[1:-1]
    return s


def extract_boxed(text):
    boxes = _boxed_spans(text)
    return norm(boxes[-1]) if boxes else None


def extract_answer(text):
    boxed = extract_boxed(text)
    if boxed is not None:
        return boxed
    nums = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return norm(nums[-1]) if nums else None


def _to_float(s):
    try:
        return float(s)
    except (TypeError, ValueError):
        return None


def _math_verify(pred, gold):
    try:
        from math_verify import parse, verify
    except ImportError:
        return False
    try:
        return bool(verify(parse("\\boxed{" + gold + "}"), parse("\\boxed{" + pred + "}")))
    except Exception:
        return False


def answers_match(pred, gold):
    if pred is None or gold is None:
        return False
    p, g = norm(pred), norm(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    gf = _to_float(g)
    if gf is not None:
        m = re.fullmatch(r"(-?\d+(?:\.\d+)?)[a-zA-Z]*", p)
        pf = _to_float(m.group(1)) if m else None
        return pf is not None and abs(pf - gf) < 1e-6
    return _math_verify(p, g)


def gsm8k_gold(answer_field):
    return norm(answer_field.split("####")[-1])


def has_boxed(text):
    return bool(_boxed_spans(text))

In [ ]:
%%writefile rf_eval.py
import os
import pickle
import random
from collections import Counter

from rf_extract import answers_match, extract_answer, extract_boxed, gsm8k_gold, has_boxed, user_prompt

DEV_SIZE = 500


def gsm8k_train(split=None, seed=0):
    from datasets import load_dataset
    ds = load_dataset("openai/gsm8k", "main", split="train")
    order = list(range(len(ds)))
    random.Random(seed).shuffle(order)
    dev = set(order[:DEV_SIZE])
    items = [{"id": i, "split": "dev" if i in dev else "train",
              "question": r["question"], "gold": gsm8k_gold(r["answer"])}
             for i, r in enumerate(ds)]
    return [it for it in items if split in (None, it["split"])]


def load_eval_set(name, n=None):
    from datasets import load_dataset
    if name == "gsm8k":
        ds = load_dataset("openai/gsm8k", "main", split="test")
        items = [{"question": r["question"], "gold": gsm8k_gold(r["answer"])} for r in ds]
    elif name == "gsm8k_dev":
        items = gsm8k_train("dev")
    elif name == "math500":
        ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
        items = [{"question": r["problem"], "gold": r["answer"],
                  "subject": r["subject"], "level": r["level"]} for r in ds]
    else:
        raise ValueError(name)
    return items[:n] if n else items


def chat_prompts(tokenizer, items, style):
    return [tokenizer.apply_chat_template([{"role": "user", "content": user_prompt(it["question"], style)}],
                                          tokenize=False, add_generation_prompt=True)
            for it in items]


def is_looping(text, tail=3000, n=8, min_repeats=5):
    words = text[-tail:].split()
    if len(words) < 50:
        return False
    grams = Counter(" ".join(words[i:i + n]) for i in range(len(words) - n))
    return grams.most_common(1)[0][1] >= min_repeats


def completion_row(c):
    return {"resp": c.text, "n_new_tokens": len(c.token_ids),
            "truncated": int(c.finish_reason == "length"),
            "has_box": has_boxed(c.text), "looping": is_looping(c.text)}


def rows_from_outputs(items, outputs):
    return [{**it, **completion_row(out.outputs[0])} for it, out in zip(items, outputs)]


def graded(c, gold):
    row = completion_row(c)
    row["pred"] = extract_boxed(c.text)
    row["reward"] = int(answers_match(row["pred"], gold))
    return row


def sample_rows(items, greedy_outputs, sampled_outputs):
    return [{**it, "greedy": graded(g.outputs[0], it["gold"]),
             "samples": [graded(c, it["gold"]) for c in s.outputs]}
            for it, g, s in zip(items, greedy_outputs, sampled_outputs)]


def majority_correct(row):
    preds = [s["pred"] for s in row["samples"] if s["pred"] is not None]
    return bool(preds) and answers_match(Counter(preds).most_common(1)[0][0], row["gold"])


def sample_summary(rows):
    n, k = len(rows), len(rows[0]["samples"])
    passes = [sum(s["reward"] for s in r["samples"]) for r in rows]
    samples = [s for r in rows for s in r["samples"]]
    tokens = sorted(s["n_new_tokens"] for s in samples)
    return {
        "n": n, "k": k,
        "greedy": sum(r["greedy"]["reward"] for r in rows) / n,
        "sample_mean": sum(passes) / (n * k),
        "majority": sum(majority_correct(r) for r in rows) / n,
        "pass_any": sum(p > 0 for p in passes) / n,
        "mixed": sum(0 < p < k for p in passes) / n,
        "hist": [passes.count(i) for i in range(k + 1)],
        "truncated": sum(s["truncated"] for s in samples) / len(samples),
        "looping": sum(s["looping"] for s in samples) / len(samples),
        "median_tokens": tokens[len(tokens) // 2],
    }


def quick_acc(rows):
    return sum(answers_match(extract_answer(r["resp"]), r["gold"]) for r in rows) / max(len(rows), 1)


def load_result(path, config):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return {"config": config}


def save_result(path, result):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(result, f)
    os.replace(tmp, path)

## 4. Config

In [ ]:
import glob, os, shutil

MODEL_ID    = "Qwen/Qwen2.5-1.5B-Instruct"
K           = 8
TEMPERATURE = 1.0
MAX_NEW     = 1024
CHUNK       = 500
N_LIMIT     = None
OUT_DIR     = "/kaggle/working/samples"
OUT         = f"{OUT_DIR}/gsm8k_train_k{K}.pkl"
os.makedirs(OUT_DIR, exist_ok=True)

for p in glob.glob("/kaggle/input/**/samples/*.pkl", recursive=True):
    dst = os.path.join(OUT_DIR, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst); print("resuming from", p)

## 5. Load vLLM

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    dtype="half",
    max_model_len=MAX_NEW + 512,
    gpu_memory_utilization=0.85,
    seed=0,
)
tokenizer = llm.get_tokenizer()

GREEDY = SamplingParams(temperature=0.0, max_tokens=MAX_NEW, repetition_penalty=1.0)
SAMPLED = SamplingParams(n=K, temperature=TEMPERATURE, top_p=1.0, max_tokens=MAX_NEW,
                         repetition_penalty=1.0, seed=0)

## 6. Generate

In [ ]:
import time
from rf_eval import gsm8k_train, chat_prompts, sample_rows, sample_summary, load_result, save_result

items = gsm8k_train()
if N_LIMIT:
    items = items[:N_LIMIT]

result = load_result(OUT, {"dataset": "gsm8k_train", "style": "boxed", "n": len(items), "k": K,
                           "temperature": TEMPERATURE, "max_new": MAX_NEW, "model_id": MODEL_ID})
rows = result.setdefault("rows", [])
print(f"{len(rows)}/{len(items)} problems already done")

t0 = time.time()
for start in range(len(rows), len(items), CHUNK):
    chunk = items[start:start + CHUNK]
    prompts = chat_prompts(tokenizer, chunk, "boxed")
    greedy = llm.generate(prompts, GREEDY)
    sampled = llm.generate(prompts, SAMPLED)
    rows.extend(sample_rows(chunk, greedy, sampled))
    save_result(OUT, result)
    s = sample_summary(rows)
    print(f"[{len(rows)}/{len(items)}] greedy {s['greedy']*100:.1f}%  pass@{K} {s['pass_any']*100:.1f}%  "
          f"mixed {s['mixed']*100:.1f}%  ({(time.time()-t0)/60:.0f} min)", flush=True)

## 7. Summary

In [ ]:
import json

summary = {}
for split in ["train", "dev"]:
    part = [r for r in rows if r["split"] == split]
    if not part:
        continue
    s = summary[split] = sample_summary(part)
    print(f"\n{split} (n={s['n']})")
    print(f"  greedy            {s['greedy']*100:5.1f}%")
    print(f"  mean of samples   {s['sample_mean']*100:5.1f}%")
    print(f"  majority@{K}        {s['majority']*100:5.1f}%")
    print(f"  pass@{K}            {s['pass_any']*100:5.1f}%")
    print(f"  mixed (1..{K-1} of {K}) {s['mixed']*100:5.1f}%")
    print(f"  truncated {s['truncated']*100:.1f}%  looping {s['looping']*100:.1f}%  median tokens {s['median_tokens']}")
    print("  correct out of", K, ":", "  ".join(f"{i}:{c}" for i, c in enumerate(s["hist"])))

with open(f"{OUT_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

## 8. Package for download

In [ ]:
shutil.make_archive("/kaggle/working/samples", "zip", "/kaggle/working", "samples")
print("download /kaggle/working/samples.zip from the Output panel")